# Geo Data Prep — Cleaned

Builds four Parquet files consumed by the `us_ai_map` Django app:

- `data/output/state_map_data.parquet` — per `(soc, task, state)` with `weight`, `a_mean`, `pi`, `penetration`, `beta`
- `data/output/msa_map_data.parquet` — same, per `(soc, task, msa-or-nonmetro)`
- `data/output/task_dim.parquet` — task id → task statement text
- `data/output/occupation_dim.parquet` — soc code → title, job family

Key optimisations vs the prior version:

1. `all_data_M_2024.csv` is read **once** (was 3×). Explicit dtypes + `usecols` + `thousands=","` kill the `DtypeWarning` and the manual `str.replace(",", "")` dance.
2. Remote CSVs (ONET, Anthropic, Eloundou, HF) are cached locally as Parquet after first fetch — reruns are offline.
3. `pi` is renormalised **once** on the 17k-row task_exposure frame (group by `soc_code`, use `transform("sum")`) instead of with a Python lambda on the 4.4M-row merged frame.
4. `.XX` occupation-code suffix is stripped once, upstream, not after the fan-out merge.
5. Output dtypes compacted (`int32`/`int8`/`float32`) so parquets are ~2× smaller and Django ingest is faster.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ONET_VERSION = "30_2"

DATA_DIR = Path("./data")
OUTPUT_DIR = DATA_DIR / "output"
CACHE_DIR = DATA_DIR / "_cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)


## Cache helper

Remote CSVs are fetched once and memoised as Parquet under `data/_cache/`. Delete the parquet to force a refresh.

In [2]:
def cached_read_csv(url: str, cache_name: str, **read_csv_kwargs) -> pd.DataFrame:
    cache_path = CACHE_DIR / f"{cache_name}.parquet"
    if cache_path.exists():
        return pd.read_parquet(cache_path)
    df = pd.read_csv(url, **read_csv_kwargs)
    df.to_parquet(cache_path, compression="snappy", index=False)
    return df


## Task exposure

Join **Anthropic penetration** (observed AI adoption per task) × **Eloundou beta** (theoretical AI exposure per task) × **ONET pi** (task's share of the occupation's labor input) × **job family**.

`beta` is clamped from 1.0 → 0.5 for the E1 class: Eloundou labels E1 as "AI accelerates ≥50%", so 0.5 is the defensible lower bound for a scalar exposure score.

In [3]:
task_statements = cached_read_csv(
    f"https://www.onetcenter.org/dl_files/database/db_{ONET_VERSION}_text/Task%20Statements.txt",
    cache_name=f"onet_task_statements_{ONET_VERSION}",
    delimiter="\t",
)[["O*NET-SOC Code", "Task ID", "Task"]]

anthropic_pen = cached_read_csv(
    "https://huggingface.co/datasets/Anthropic/EconomicIndex/resolve/main/labor_market_impacts/task_penetration.csv",
    cache_name="anthropic_task_penetration",
)

# Anthropic's dataset is keyed by task text, not ONET task id — bridge via task_statements
anthropic_pen = (
    task_statements.merge(anthropic_pen, left_on="Task", right_on="task", how="inner")
    .drop(columns=["Task", "task"])
    .drop_duplicates()
)

eloundou = cached_read_csv(
    "https://raw.githubusercontent.com/openai/GPTs-are-GPTs/refs/heads/main/data/full_labelset.tsv",
    cache_name="eloundou_full_labelset",
    delimiter="\t",
)[["O*NET-SOC Code", "Task ID", "beta"]]
# E1 (beta==1): AI accelerates task by ≥50% — clamp to the 0.5 lower bound.
eloundou.loc[np.isclose(eloundou["beta"], 1.0), "beta"] = 0.5

job_families = (
    pd.read_csv(DATA_DIR / "ONET" / "All_Job_Families.csv")
    .drop(columns=["Occupation"])
    .rename(columns={"Code": "onetsoc_code", "Job Family": "job_family"})
)

onet_weights = cached_read_csv(
    f"https://huggingface.co/datasets/MIT-WAL/job_task_input_share/resolve/main/task_labor_input_mean_estimates/{ONET_VERSION}/ONET_{ONET_VERSION}_weight_mode_STANDARD_task_labor_input_mean_estimates.csv",
    cache_name=f"onet_task_labor_input_{ONET_VERSION}",
)

task_exposure = (
    anthropic_pen.merge(eloundou, on=["O*NET-SOC Code", "Task ID"], how="inner")
    .rename(columns={"O*NET-SOC Code": "onetsoc_code", "Task ID": "task_id"})
    .merge(onet_weights, on=["onetsoc_code", "task_id"], how="inner")
    .merge(job_families, on="onetsoc_code", how="left")
)


In [4]:
# Drop detailed occupations
print(f"task_exposure: {len(task_exposure):,} rows, {task_exposure['onetsoc_code'].nunique():,} soc codes")
task_exposure["detail_column"] = task_exposure["onetsoc_code"].str.split(".").str[1].astype(int)
task_exposure = task_exposure[task_exposure["detail_column"]==0].copy()

# Strip ".XX" suffix once, before the fan-out merges.
task_exposure["onetsoc_code"] = task_exposure["onetsoc_code"].str.replace(r"\.\d+$", "", regex=True)
task_exposure = task_exposure.rename(columns={"onetsoc_code": "soc_code"})

# Renormalise pi so it sums to 1 per soc_code (the three-way inner merge drops some tasks).
# Doing it here on ~17k rows avoids the 4.4M-row groupby.transform(lambda) used previously.
task_exposure["pi"] = task_exposure["pi"] / task_exposure.groupby("soc_code")["pi"].transform("sum")

task_exposure = task_exposure.astype({"task_id": "int32"})
print(f"task_exposure: {len(task_exposure):,} rows, {task_exposure['soc_code'].nunique():,} soc codes")
task_exposure.head(2)


task_exposure: 17,220 rows, 894 soc codes
task_exposure: 14,176 rows, 748 soc codes


,soc_code,task_id,penetration,beta,pi,job_family,detail_column
0,11-1011,8823,0.0,0.5,0.047235,Management,0
1,11-1011,8824,0.0,0.0,0.048477,Management,0


## OEWS — single read

One pass over `all_data_M_2024.csv` with explicit dtypes; every downstream slice filters this frame. `thousands=","` replaces the manual comma-stripping, `na_values` handles OEWS's `*` / `#` sentinels.

In [5]:
_OEWS_USECOLS = ["AREA", "AREA_TYPE", "OCC_CODE", "OCC_TITLE", "O_GROUP", "TOT_EMP", "A_MEAN"]
_OEWS_DTYPES = {
    "AREA": "int32",
    "AREA_TYPE": "int8",
    "OCC_CODE": "string",
    "OCC_TITLE": "string",
    "O_GROUP": "string",
}

oews_all = pd.read_csv(
    DATA_DIR / "OEWS" / "all_data_M_2024.csv",
    usecols=_OEWS_USECOLS,
    dtype=_OEWS_DTYPES,
    thousands=",",
    na_values=["*", "#", "**"],
)

oews_detailed = oews_all.loc[oews_all["O_GROUP"] == "detailed"].copy()
oews_detailed[["TOT_EMP", "A_MEAN"]] = oews_detailed[["TOT_EMP", "A_MEAN"]].fillna(0)

print(f"OEWS rows: total {len(oews_all):,}, detailed {len(oews_detailed):,}")


OEWS rows: total 414,437, detailed 307,153


## Geography payload builder

Reused for both state (AREA_TYPE ∈ {2, 3}) and metro/nonmetro (AREA_TYPE ∈ {4, 6}). Multiplying `weight` by `pi` (already renormalised) redistributes each occupation's area-level employment across its tasks.

In [6]:
def build_geo_payload(area_types, code_col, include_area_type):
    cols = ["AREA", "OCC_CODE", "TOT_EMP", "A_MEAN"]
    if include_area_type:
        cols.append("AREA_TYPE")
    geo = oews_detailed.loc[oews_detailed["AREA_TYPE"].isin(area_types), cols].rename(columns={
        "AREA": code_col,
        "OCC_CODE": "soc_code",
        "TOT_EMP": "weight",
        "A_MEAN": "a_mean",
        "AREA_TYPE": "area_type",
    })

    audit_workforce = float(geo["weight"].sum())
    audit_wage_bill = float((geo["weight"] * geo["a_mean"]).sum())

    merged = task_exposure.merge(geo, on="soc_code", how="inner")
    merged["weight"] = merged["weight"] * merged["pi"]

    kept_workforce = float(merged["weight"].sum())
    kept_wage_bill = float((merged["weight"] * merged["a_mean"]).sum())
    print(
        f"  coverage: workforce {kept_workforce / audit_workforce:.2%} "
        f"(of {audit_workforce:,.0f}), wage bill {kept_wage_bill / audit_wage_bill:.2%}"
    )

    dtype_map = {code_col: "int32", "weight": "float32", "a_mean": "float32",
                 "penetration": "float32", "beta": "float32", "pi": "float32"}
    if include_area_type:
        dtype_map["area_type"] = "int8"
    return merged.astype(dtype_map)


## State payload

In [7]:
print("state (AREA_TYPE 2, 3):")
state_map = build_geo_payload([2, 3], code_col="state_code", include_area_type=False)
# state_code values are FIPS 1..56 — tighten to int8
state_map["state_code"] = state_map["state_code"].astype("int8")
state_map.to_parquet(OUTPUT_DIR / "state_map_data.parquet", compression="snappy", index=False)
print(f"  wrote {OUTPUT_DIR / 'state_map_data.parquet'} ({len(state_map):,} rows)")
state_map.head(2)


state (AREA_TYPE 2, 3):
  coverage: workforce 87.72% (of 154,059,520), wage bill 87.30%
  wrote data/output/state_map_data.parquet (618,825 rows)


,soc_code,task_id,penetration,beta,pi,job_family,detail_column,state_code,weight,a_mean
0,11-1011,8823,0.0,0.5,0.047235,Management,0,1,39.205315,207190.0
1,11-1011,8823,0.0,0.5,0.047235,Management,0,2,42.984142,211500.0


## Metro + Nonmetro payload

In [8]:
print("metro + nonmetro (AREA_TYPE 4, 6):")
msa_map = build_geo_payload([4, 6], code_col="msa_code", include_area_type=True)
msa_map.to_parquet(OUTPUT_DIR / "msa_map_data.parquet", compression="snappy", index=False)
print(f"  wrote {OUTPUT_DIR / 'msa_map_data.parquet'} ({len(msa_map):,} rows)")
msa_map.head(2)


metro + nonmetro (AREA_TYPE 4, 6):
  coverage: workforce 87.77% (of 144,965,680), wage bill 87.39%
  wrote data/output/msa_map_data.parquet (3,330,688 rows)


,soc_code,task_id,penetration,beta,pi,job_family,detail_column,msa_code,weight,a_mean,area_type
0,11-1011,8823,0.0,0.5,0.047235,Management,0,10180,2.361766,213250.0,4
1,11-1011,8823,0.0,0.5,0.047235,Management,0,10380,5.668238,100550.0,4


## Dimension tables

- `task_dim`: every `(Task ID, Task)` pair from the ONET task statements. Columns preserved as `Task ID` / `Task` to match the existing DB loader.
- `occupation_dim`: every soc code that ends up in the map data, joined to its OEWS title and ONET job family.

In [9]:
task_dim = (
    task_statements[["Task ID", "Task"]]
    .drop_duplicates()
    .astype({"Task ID": "int32"})
)
task_dim = task_dim[task_dim["Task ID"].isin(task_exposure["task_id"])].copy()
task_dim.to_parquet(OUTPUT_DIR / "task_dim.parquet", compression="snappy", index=False)
print(f"task_dim: {len(task_dim):,} rows")

occ_titles = (
    oews_detailed[["OCC_CODE", "OCC_TITLE"]]
    .drop_duplicates()
    .rename(columns={"OCC_CODE": "onetsoc_code", "OCC_TITLE": "onetsoc_title"})
)

occupation_dim = (
    task_exposure[["soc_code", "job_family"]]
    .drop_duplicates()
    .rename(columns={"soc_code": "onetsoc_code"})
    .merge(occ_titles, on="onetsoc_code", how="inner")
    [["onetsoc_code", "onetsoc_title", "job_family"]]
)
occupation_dim.to_parquet(OUTPUT_DIR / "occupation_dim.parquet", compression="snappy", index=False)
print(f"occupation_dim: {len(occupation_dim):,} rows")

task_dim: 14,176 rows
occupation_dim: 725 rows


## Cleanup

Free the OEWS frame so a long-running kernel doesn't accumulate hundreds of MB.

In [10]:
del oews_all, oews_detailed
